In [1]:
import pytesseract
import cv2
import pandas as pd
from pathlib import Path

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

processed_dir = Path("../data/processed")

In [2]:
img = cv2.imread(str(processed_dir / "receipt_01.jpg"))

config = "--psm 6 -l eng"
text = pytesseract.image_to_string(img, config=config)
print("=== PSM 6 OUTPUT ===")
print(text)

=== PSM 6 OUTPUT ===
° -ELEYE,
Neribrog Ine, .
° Owned ¢ Operated by: Ner tbrog
Ine,
VATREGTIN ¥479-423-988 000
Unit 4 3 9, Rea] Nare Bul ld ing,
: Pinikitan, Cagayan De Org City,
Misamis Orental, Phi pp inigs
Tal 2: (02) 0000009
05/10/2026 (sun) 17:06:05
. INVOICE #1456745 RESET_cNTyo
STORE#2006 SN# 9200064.
MIN gs 2106171638 1259046
[STAFF Apr qy Rose J, Zabal lero
NaturesPur ipwsogm 7 15,00y
nd Me Chips 989 35,00
Tota] (2) 50.00
CASH 50.00
CHANGE 0.00
Vatable 44,64
VAT_Amt 5.36
Zero_Rated Sales 0,00
VAT Exempt Sales 0,00
Loyalty No: —
dhs
rH
eee
Philippine Seven Corporation
7th Floor The Columbia Tower
Ortigas Avenue, Mandaluyong
City
, TIN: 000-390- 189-999
BIR Acer #
|16-000380189-c0a346- 19599
Date Issued: 08/01/2020
PIU ge
*P062021-098-0293524-aggqy
SAAN MAN PATUNGO KASAMA Mo ANG
IYONG PABORTTONG KAPTTBAHAY
BUY P2009 WORTH OF ITEMS GET A
: CHANCE Tg WIN KAPTTBIYAHE
TRAVEL REWARDS Per DTI FAIR
TRADE Permit Number: 251660
| Serfes of
2026. facebook com/7 1 Iphil} Paine
S,
- THIS I

In [3]:
img = cv2.imread(str(processed_dir / "receipt_01.jpg"))

for psm in [3, 4, 6]:
    config = f"--psm {psm} -l eng"
    text = pytesseract.image_to_string(img, config=config)
    print(f"\n=== PSM {psm} ===")
    print(text[:300])


=== PSM 3 ===
¢-ELEVEnp,

Neribrog Ine,
Owned ¢ Operated by: Neribros

Ine,
VATREGTIN ¥479-423-988 000
Unit 4 9, Real Nare Bul ld ing,
Pinikitan, Cagayan De Org City,
Misamis Orental, Phi pp inigs
Tal 2: (02) 0000009

05/10/2026 (sun) 17:06:05
INVOICE #1456745 RESET_cNTyo
STORE#2006 SN# 9200064.
MIN 3B. 210617163

=== PSM 4 ===
¢-ELEVEnp,

Neribrog Ine,
Owned ¢ Operated by: Neribros

Ine,

VATREGTIN ¥479-423-988 000

Unit 4 9, Real Nare Bul ld ing,

Pinikitan, Cagayan De Org City,

Misamis Orental, Phi pp inigs
Tal 2: (02) 0000009

05/10/2026 (sun) 1

INVOICE #1456745 RESET_cntuo
STORE#2006

SN# 9200064.
MIN 3B. 2106171638

=== PSM 6 ===
° -ELEYE,
Neribrog Ine, .
° Owned ¢ Operated by: Ner tbrog
Ine,
VATREGTIN ¥479-423-988 000
Unit 4 3 9, Rea] Nare Bul ld ing,
: Pinikitan, Cagayan De Org City,
Misamis Orental, Phi pp inigs
Tal 2: (02) 0000009
05/10/2026 (sun) 17:06:05
. INVOICE #1456745 RESET_cNTyo
STORE#2006 SN# 9200064.
MIN gs 210


In [4]:
img = cv2.imread(str(processed_dir / "receipt_01.jpg"))
config = "--psm 6 -l eng"

data = pytesseract.image_to_data(img, config=config, output_type=pytesseract.Output.DATAFRAME)
data = data[data["conf"] > 0]

print(data[["text", "conf"]].dropna())
print(f"\nAverage confidence: {data['conf'].mean():.2f}")

         text       conf
4           °  24.604126
7    Neribrog  26.970474
8        Ine,  91.600754
11          °  35.196068
12      Owned  81.020439
..        ...        ...
212      THIS  91.195633
213        IS  67.723152
214        an  94.770874
215   INVOICE  89.608582
216         -  86.422646

[153 rows x 2 columns]

Average confidence: 67.70


In [5]:
import pandas as pd

df = pd.read_csv("../data/annotated/ground_truth.csv")
results = []

for _, row in df.iterrows():
    img = cv2.imread(str(processed_dir / row["filename"]))
    if img is None:
        print(f"Could not read {row['filename']}")
        continue

    config = "--psm 6 -l eng"
    text = pytesseract.image_to_string(img, config=config)
    data = pytesseract.image_to_data(img, config=config, output_type=pytesseract.Output.DATAFRAME)
    data = data[data["conf"] > 0]
    avg_conf = data["conf"].mean()

    results.append({
        "filename": row["filename"],
        "condition": row["condition"],
        "ocr_text": text.strip(),
        "avg_confidence": round(avg_conf, 2)
    })
    print(f"Done: {row['filename']} | Avg confidence: {avg_conf:.2f}")

results_df = pd.DataFrame(results)
results_df.to_csv("../outputs/metrics/ocr_raw_results.csv", index=False)
print("\nAll results saved to outputs/metrics/ocr_raw_results.csv")

Done: receipt_01.jpg | Avg confidence: 67.70
Done: receipt_02.jpg | Avg confidence: 68.65
Done: receipt_03.jpg | Avg confidence: 78.77
Done: receipt_04.jpg | Avg confidence: 62.21
Done: receipt_05.jpg | Avg confidence: 79.13
Done: receipt_06.jpg | Avg confidence: 54.18
Done: receipt_07.jpg | Avg confidence: 64.42
Done: receipt_08.jpg | Avg confidence: 61.23
Done: receipt_09.jpg | Avg confidence: 57.04
Done: receipt_10.jpg | Avg confidence: 65.91
Done: receipt_11.jpg | Avg confidence: 69.54
Done: receipt_12.jpg | Avg confidence: 65.95
Done: receipt_13.jpg | Avg confidence: 67.61
Done: receipt_14.jpg | Avg confidence: 43.56
Done: receipt_15.jpg | Avg confidence: 66.67
Done: receipt_16.jpg | Avg confidence: 62.72
Done: receipt_17.jpg | Avg confidence: 64.56

All results saved to outputs/metrics/ocr_raw_results.csv


In [6]:
img = cv2.imread(str(processed_dir / "receipt_01.jpg"))
config = "--psm 6 -l eng"
text = pytesseract.image_to_string(img, config=config)
print(text[:400])

° -ELEYE,
Neribrog Ine, .
° Owned ¢ Operated by: Ner tbrog
Ine,
VATREGTIN ¥479-423-988 000
Unit 4 3 9, Rea] Nare Bul ld ing,
: Pinikitan, Cagayan De Org City,
Misamis Orental, Phi pp inigs
Tal 2: (02) 0000009
05/10/2026 (sun) 17:06:05
. INVOICE #1456745 RESET_cNTyo
STORE#2006 SN# 9200064.
MIN gs 2106171638 1259046
[STAFF Apr qy Rose J, Zabal lero
NaturesPur ipwsogm 7 15,00y
nd Me Chips 989 35,00
T


In [7]:
for i in range(1, 30):
    filename = f"receipt_{i:02d}.jpg"
    img = cv2.imread(str(processed_dir / filename))
    
    if img is None:  # skip if file doesn't exist
        continue
    
    config = "--psm 6 -l eng"
    text = pytesseract.image_to_string(img, config=config)
    
    # Extract just the total line
    for line in text.split('\n'):
        if 'Total' in line or 'TOTAL' in line or 'Amount Due' in line:
            print(f"{filename} → {line.strip()}")
            break

receipt_02.jpg → Total ¢ 1) 203.00
receipt_03.jpg → Total (2) 44,00
receipt_04.jpg → Total amount Due ap) 24.00
receipt_06.jpg → Total] Amouryt + 2000
receipt_07.jpg → Tata) Amount Due (1) 11.00
receipt_10.jpg → Total Amount Due (2) 81.00
receipt_11.jpg → , Total Amount Due (1) 59.00 fo
receipt_13.jpg → Total Amount Bue (23 22.00
receipt_14.jpg → 7 Total O) 8.00
receipt_15.jpg → Total Amount Due (7) 38.00 ee !
receipt_16.jpg → Total Amount Due (2) 50.00 |.
receipt_17.jpg → Total Amount Due (1) 59.00 . —_—
